In [7]:
import os
import re
from pathlib import Path
from urllib.parse import urlparse, parse_qs
import pandas as pd
from dotenv import load_dotenv
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# Compatible with VS Code using either the repository root or notebooks/ as cwd.
cwd = Path.cwd().resolve()
root_candidates = [cwd, *list(cwd.parents)]
PROJECT_ROOT = next((p for p in root_candidates if (p / "requirements.txt").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot find repository root containing requirements.txt")

ENV_FILE = PROJECT_ROOT / ".env"
if not ENV_FILE.is_file():
    raise FileNotFoundError(f"Create a private .env file at {ENV_FILE}")
load_dotenv(ENV_FILE, override=True)
API_KEY = os.getenv("YOUTUBE_API_KEY", "").strip()
if not API_KEY:
    raise ValueError("YOUTUBE_API_KEY is missing from .env")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
ANNOT_DIR = PROJECT_ROOT / "data" / "annotated"
RAW_DIR.mkdir(parents=True, exist_ok=True)
ANNOT_DIR.mkdir(parents=True, exist_ok=True)
print("Repository:", PROJECT_ROOT.name)
print("API key found: True (value hidden)")

Repository: FYP-Vietnamese-Sarcasm
API key found: True (value hidden)


In [8]:
youtube = build("youtube", "v3", developerKey=API_KEY, cache_discovery=False)
try:
    test = youtube.videos().list(part="snippet", id="jNQXAC9IVRw").execute()
    print("YouTube API connected.", "Test videos returned:", len(test.get("items", [])))
except HttpError as exc:
    print("API error:", getattr(exc.resp, "status", "unknown"))
    raise

YouTube API connected. Test videos returned: 1


In [2]:
VIDEO_URLS = [
     "https://youtu.be/lQvsfDY6ef8",
    "https://youtu.be/fyrd-mpWxig",
]

MAX_COMMENTS_PER_VIDEO = 60  # Keep this small for a pilot.
PILOT_SAMPLE_SIZE = 40       # 30–50 is a useful second-pilot target.
RANDOM_SEED = 42

def video_id_from_url(value):
    s = value.strip()
    if re.fullmatch(r"[A-Za-z0-9_-]{11}", s):
        return s
    u = urlparse(s)
    host = (u.hostname or "").lower()
    if host in {"youtu.be", "www.youtu.be"}:
        candidate = u.path.strip("/").split("/")[0]
    elif host in {"youtube.com", "www.youtube.com", "m.youtube.com"}:
        if u.path == "/watch":
            candidate = parse_qs(u.query).get("v", [""])[0]
        elif u.path.startswith(("/shorts/", "/live/", "/embed/")):
            candidate = u.path.split("/")[2]
        else:
            candidate = ""
    else:
        candidate = ""
    if not re.fullmatch(r"[A-Za-z0-9_-]{11}", candidate):
        raise ValueError(f"Invalid YouTube video URL/ID: {value}")
    return candidate

VIDEO_IDS = list(dict.fromkeys(video_id_from_url(u) for u in VIDEO_URLS))
print("Videos selected:", len(VIDEO_IDS))

NameError: name 're' is not defined